# Portada de Entrega
Universidad del Valle de Guatemala
Inteligencia Artificial - Laboratorio 02
Javier España
Ángel Esquit
Roberto Barreda

Repositorio: https://github.com/Javier-Espana/Lab02-IA.git

# Clasificación Play Golf

In [7]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression

In [8]:
# Datos del enunciado
data = [
    {"Outlook": "Rainy",    "Temperature": "Hot",  "Humidity": "High",   "Windy": "False", "Play": "No"},
    {"Outlook": "Rainy",    "Temperature": "Hot",  "Humidity": "High",   "Windy": "True",  "Play": "No"},
    {"Outlook": "Overcast", "Temperature": "Hot",  "Humidity": "High",   "Windy": "False", "Play": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Mild", "Humidity": "High",   "Windy": "False", "Play": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Cool", "Humidity": "Normal", "Windy": "False", "Play": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Cool", "Humidity": "Normal", "Windy": "True",  "Play": "No"},
    {"Outlook": "Overcast", "Temperature": "Cool", "Humidity": "Normal", "Windy": "True",  "Play": "Yes"},
    {"Outlook": "Rainy",    "Temperature": "Mild", "Humidity": "High",   "Windy": "False", "Play": "No"},
    {"Outlook": "Rainy",    "Temperature": "Cool", "Humidity": "Normal", "Windy": "False", "Play": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Mild", "Humidity": "Normal", "Windy": "False", "Play": "Yes"},
    {"Outlook": "Rainy",    "Temperature": "Mild", "Humidity": "Normal", "Windy": "True",  "Play": "Yes"},
    {"Outlook": "Overcast", "Temperature": "Mild", "Humidity": "High",   "Windy": "True",  "Play": "Yes"},
    {"Outlook": "Overcast", "Temperature": "Hot",  "Humidity": "Normal", "Windy": "False", "Play": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Mild", "Humidity": "High",   "Windy": "True",  "Play": "No"}
]
df = pd.DataFrame(data)
df

,Outlook,Temperature,Humidity,Windy,Play
0,Rainy,Hot,High,False,No
1,Rainy,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Sunny,Mild,High,False,Yes
4,Sunny,Cool,Normal,False,Yes
5,Sunny,Cool,Normal,True,No
6,Overcast,Cool,Normal,True,Yes
7,Rainy,Mild,High,False,No
8,Rainy,Cool,Normal,False,Yes
9,Sunny,Mild,Normal,False,Yes


## Modelos y validacion
Usamos One-Hot Encoding y Leave-One-Out (LOO) para evaluar con muy pocos datos.

In [9]:
X = df.drop(columns=["Play"])
y = df["Play"].map({"No": 0, "Yes": 1})

categorical_features = X.columns.tolist()
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

models = {
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "Naive Bayes (Bernoulli)": BernoulliNB(),
    "Regresion Logistica": LogisticRegression(solver="liblinear", max_iter=1000)
}

loo = LeaveOneOut()

def evaluate_model(model):
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    y_pred = cross_val_predict(pipe, X, y, cv=loo, method="predict")
    y_proba = cross_val_predict(pipe, X, y, cv=loo, method="predict_proba")[:, 1]

    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_proba),
    }

results = {name: evaluate_model(model) for name, model in models.items()}
pd.DataFrame(results).T.sort_values(by="f1", ascending=False)

,accuracy,precision,recall,f1,roc_auc
Regresion Logistica,0.571429,0.636364,0.777778,0.700000,0.555556
Naive Bayes (Bernoulli),0.500000,0.625000,0.555556,0.588235,0.533333
KNN (k=3),0.500000,0.666667,0.444444,0.533333,0.477778


## Prediccion de casos nuevos
x1 = (Rainy, Hot, High, False)
x2 = (Sunny, Hot, Normal, False)

In [10]:
new_cases = pd.DataFrame([
    {"Outlook": "Rainy", "Temperature": "Hot", "Humidity": "High", "Windy": "False"},
    {"Outlook": "Sunny", "Temperature": "Hot", "Humidity": "Normal", "Windy": "False"},
])

pred_table = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipe.fit(X, y)
    proba_yes = pipe.predict_proba(new_cases)[:, 1]
    pred = (proba_yes >= 0.5).astype(int)

    for i, p in enumerate(proba_yes):
        pred_table.append({
            "modelo": name,
            "caso": f"x{i+1}",
            "prob_yes": float(p),
            "prediccion": "Yes" if pred[i] == 1 else "No"
        })

pd.DataFrame(pred_table)

,modelo,caso,prob_yes,prediccion
0,KNN (k=3),x1,0.333333,No
1,KNN (k=3),x2,1.000000,Yes
2,Naive Bayes (Bernoulli),x1,0.158440,No
3,Naive Bayes (Bernoulli),x2,0.907081,Yes
4,Regresion Logistica,x1,0.387352,No
5,Regresion Logistica,x2,0.734669,Yes


## ¿Cómo construir un buen clasificador con pocos datos?
- Preferir modelos simples con regularizacion (menos varianza).
- Usar validacion cruzada (LOO) para aprovechar todo el dataset.
- Incluir conocimiento del dominio y buenas variables.
- Evitar tuning agresivo que sobreajuste.
- Si es posible, recolectar mas datos o usar datos sinteticos con cuidado.